In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
from IPython.display import display

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/GlycemIA/EDA/data/OhioT1DM")

In [ ]:
xml_files = list(DATA_DIR.rglob("*.xml"))
print(f"Total de xml files: {len(xml_files)}")

In [ ]:
inventory = []
for file in xml_files:
    file_name = file.name
    parent_folder = file.parent.name  # Train or Test Directory
    release_folder = file.parent.parent.name  # 2018 or 2020

    inventory.append(
        {
            "Patient_File": file_name,
            "Release": release_folder,
            "Folder": parent_folder,
            "Path": str(file),
        }
    )

if len(xml_files) > 0:
    df_inventory = pd.DataFrame(inventory).sort_values(by=["Release", "Patient_File"])
    display(df_inventory)
else:
    print(
        "ALERT: No se encontraron archivos XML. Verifica que la ruta DATA_DIR coincida exactamente con los nombres de tus carpetas en Drive."
    )

In [ ]:
def get_xml_signals(filepath):
    """
    A quick parser to return the tags of the first level of the xml file.
    """
    signals = set()
    try:
        context = ET.iterparse(filepath, events=("start",))
        _, root = next(context)

        for event, elem in context:
            if elem.tag != root.tag:
                signals.add(elem.tag)
            root.clear()
    except Exception as e:
        print(f"Error leyendo {filepath}: {e}")

    return list(signals)

In [ ]:
schema_data = []
for idx, row in df_inventory.iterrows():
    # Extract ID from the pacient's name(eg. '559-ws-training.xml' -> '559')
    patient_id = row["Patient_File"].split("-")[0]
    is_train = (
        "train" in row["Patient_File"].lower() or "train" in row["Folder"].lower()
    )
    split_type = "Train" if is_train else "Test"

    signals = get_xml_signals(row["Path"])

    patient_schema = {
        "Release": row["Release"],
        "Patient_ID": patient_id,
        "Split": split_type,
    }
    for sig in signals:
        patient_schema[sig] = True

    schema_data.append(patient_schema)

In [ ]:
df_schema = pd.DataFrame(schema_data).fillna(False)

In [ ]:
cols = df_schema.columns.tolist()
metadata_cols = ["Release", "Patient_ID", "Split"]
signal_cols = sorted([c for c in cols if c not in metadata_cols])
df_schema = df_schema[metadata_cols + signal_cols]

print("Matriz de Señales Disponibles por Paciente y Split:")
display(df_schema.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
coverage_data = []
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    # Parse the XML file
    tree = ET.parse(row["Path"])
    root = tree.getroot()

    # Find the main glucose node
    glucose_node = root.find("glucose_level")
    timestamps = []

    if glucose_node is not None:
        for event in glucose_node.findall("event"):
            ts = event.get("ts")
            if ts:
                timestamps.append(ts)

    if not timestamps:
        continue

    # Convert to datetime and wrap in a Pandas Series to allow reset_index
    df_ts = (
        pd.Series(pd.to_datetime(timestamps, dayfirst=True))
        .sort_values()
        .reset_index(drop=True)
    )

    if len(df_ts) == 0:
        continue

    start_date = df_ts.iloc[0]
    end_date = df_ts.iloc[-1]

    # Total duration in days
    duration_days = (end_date - start_date).total_seconds() / (24 * 3600)

    # Calculate gaps (missing sensor transmissions)
    # Get the difference in minutes between consecutive records
    time_diffs = df_ts.diff().dt.total_seconds() / 60.0

    # Nominal frequency = 5 mins. We consider a "gap" any jump greater than 6 mins.
    # The "lost" time is the duration of those jumps minus the 5 mins that should have passed.
    gaps = time_diffs[time_diffs > 6.0]
    total_gap_minutes = (gaps - 5.0).sum()

    # Percentage of lost time
    total_expected_minutes = duration_days * 24 * 60
    gap_percentage = (
        (total_gap_minutes / total_expected_minutes) * 100
        if total_expected_minutes > 0
        else 0
    )

    coverage_data.append(
        {
            "Release": row["Release"],
            "Patient_ID": patient_id,
            "Split": split_type,
            "Start_Date": start_date,
            "End_Date": end_date,
            "Duration_Days": round(duration_days, 1),
            "Gaps_Total_Hrs": round(total_gap_minutes / 60, 1),
            "Gap_Percentage_%": round(gap_percentage, 2),
        }
    )

In [ ]:
df_coverage = pd.DataFrame(coverage_data)

print("\n--- Temporal Coverage and CGM Gaps ---")
display(df_coverage.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
cgm_stats_data = []

In [ ]:
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    # Parse the XML file
    tree = ET.parse(row["Path"])
    root = tree.getroot()

    glucose_node = root.find("glucose_level")
    records = []

    if glucose_node is not None:
        for event in glucose_node.findall("event"):
            ts = event.get("ts")
            val = event.get("value")
            if ts and val:
                records.append({"ts": ts, "glucose": float(val)})

    if not records:
        continue

    df_cgm = pd.DataFrame(records)
    df_cgm["ts"] = pd.to_datetime(df_cgm["ts"], dayfirst=True)
    df_cgm = df_cgm.sort_values("ts")

    # 1. Check for duplicate timestamps
    duplicates = df_cgm.duplicated(subset=["ts"]).sum()

    # 2. Check for physiologically impossible values (< 0 or > 600)
    invalid_mask = (df_cgm["glucose"] < 0) | (df_cgm["glucose"] > 600)
    invalid_count = invalid_mask.sum()

    # Filter out invalids for statistical calculations
    df_valid = df_cgm[~invalid_mask]

    if len(df_valid) == 0:
        continue

    # 3. Calculate clinical metrics
    # Target Range: 70 - 180 mg/dL
    tir_mask = (df_valid["glucose"] >= 70) & (df_valid["glucose"] <= 180)
    tbr_mask = df_valid["glucose"] < 70  # Time Below Range (Hypoglycemia)
    tar_mask = df_valid["glucose"] > 180  # Time Above Range (Hyperglycemia)

    total_valid = len(df_valid)
    tir_pct = (tir_mask.sum() / total_valid) * 100
    tbr_pct = (tbr_mask.sum() / total_valid) * 100
    tar_pct = (tar_mask.sum() / total_valid) * 100

    # 4. Global distribution stats
    mean_gl = df_valid["glucose"].mean()
    std_gl = df_valid["glucose"].std()
    min_gl = df_valid["glucose"].min()
    max_gl = df_valid["glucose"].max()

    cgm_stats_data.append(
        {
            "Release": row["Release"],
            "Patient_ID": patient_id,
            "Split": split_type,
            "Total_Records": total_valid,
            "Duplicates": duplicates,
            "Invalid_Outliers": invalid_count,
            "Mean_Glucose": round(mean_gl, 1),
            "Std_Dev": round(std_gl, 1),
            "Min": min_gl,
            "Max": max_gl,
            "TIR_70_180_%": round(tir_pct, 1),
            "TBR_<70_%": round(tbr_pct, 1),
            "TAR_>180_%": round(tar_pct, 1),
        }
    )

In [ ]:
df_cgm_stats = pd.DataFrame(cgm_stats_data)

# Reorder logically: Metadata, Stats, Clinical Metrics
cols = [
    "Release",
    "Patient_ID",
    "Split",
    "Total_Records",
    "Duplicates",
    "Invalid_Outliers",
    "Min",
    "Max",
    "Mean_Glucose",
    "Std_Dev",
    "TBR_<70_%",
    "TIR_70_180_%",
    "TAR_>180_%",
]
df_cgm_stats = df_cgm_stats[cols]

print("\n--- CGM Statistics and Clinical Metrics (Target) ---")
display(df_cgm_stats.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
basal_stats_data = []
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    # --- Regular Basal ---
    basal_node = root.find("basal")
    basal_records = []
    if basal_node is not None:
        for event in basal_node.findall("event"):
            val = event.get("value")
            if val:
                basal_records.append(float(val))

    # --- Temporary Basal ---
    temp_basal_node = root.find("temp_basal")
    temp_basal_count = 0
    if temp_basal_node is not None:
        temp_basal_count = len(temp_basal_node.findall("event"))

    # Calculate basic statistics if regular basal exists
    if basal_records:
        basal_series = pd.Series(basal_records)
        mean_basal = basal_series.mean()
        max_basal = basal_series.max()
        min_basal = basal_series.min()
        total_basal_events = len(basal_records)
    else:
        mean_basal, max_basal, min_basal, total_basal_events = (0, 0, 0, 0)

    basal_stats_data.append(
        {
            "Release": row["Release"],
            "Patient_ID": patient_id,
            "Split": split_type,
            "Basal_Events": total_basal_events,
            "Min_Basal_(U/h)": round(min_basal, 3),
            "Mean_Basal_(U/h)": round(mean_basal, 3),
            "Max_Basal_(U/h)": round(max_basal, 3),
            "Temp_Basal_Events": temp_basal_count,
        }
    )

In [ ]:
df_basal = pd.DataFrame(basal_stats_data)

print("\n--- Basal and Temp Basal Statistics ---")
display(df_basal.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
bolus_stats_data = []
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    bolus_node = root.find("bolus")
    bolus_records = []
    bolus_types = {}

    if bolus_node is not None:
        for event in bolus_node.findall("event"):
            dose = event.get("dose")
            b_type = event.get("type", "unknown")

            if dose:
                bolus_records.append(float(dose))
                bolus_types[b_type] = bolus_types.get(b_type, 0) + 1

    total_bolus_events = len(bolus_records)
    mean_bolus = pd.Series(bolus_records).mean() if total_bolus_events > 0 else 0
    max_bolus = max(bolus_records) if total_bolus_events > 0 else 0

    # Extracting subtypes based on standard pump reporting
    normal_count = bolus_types.get("normal", 0)
    square_count = bolus_types.get("square", 0)
    dual_count = bolus_types.get("dual/square", 0)

    bolus_stats_data.append(
        {
            "Release": row["Release"],
            "Patient_ID": patient_id,
            "Split": split_type,
            "Total_Bolus_Events": total_bolus_events,
            "Mean_Bolus_Dose_(U)": round(mean_bolus, 2),
            "Max_Bolus_Dose_(U)": round(max_bolus, 2),
            "Normal_Bolus": normal_count,
            "Extended_Bolus": square_count + dual_count,
        }
    )

In [ ]:
df_bolus = pd.DataFrame(bolus_stats_data)

print("\n--- Bolus Insulin Statistics ---")
display(df_bolus.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
meal_stats_data = []

In [ ]:
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    # 1. Extract Bolus Timestamps to find the nearest one later
    bolus_node = root.find("bolus")
    bolus_ts = []
    if bolus_node is not None:
        for event in bolus_node.findall("event"):
            ts = event.get("ts")
            if ts:
                bolus_ts.append(pd.to_datetime(ts, dayfirst=True))

    df_bolus = pd.DataFrame({"bolus_ts": bolus_ts})
    if not df_bolus.empty:
        df_bolus = df_bolus.sort_values("bolus_ts")

    # 2. Extract Meal Timestamps and Carbs
    meal_node = root.find("meal")
    meals = []
    if meal_node is not None:
        for event in meal_node.findall("event"):
            ts = event.get("ts")
            carbs = event.get("carbs")
            if ts:
                meals.append(
                    {
                        "meal_ts": pd.to_datetime(ts, dayfirst=True),
                        "carbs": float(carbs) if carbs else np.nan,
                    }
                )

    df_meals = pd.DataFrame(meals)

    if df_meals.empty:
        continue

    df_meals = df_meals.sort_values("meal_ts")

    # Fetch duration from df_coverage for frequency calculations
    duration_days = df_coverage.loc[
        (df_coverage["Patient_ID"] == patient_id)
        & (df_coverage["Split"] == split_type),
        "Duration_Days",
    ].values
    duration_days = duration_days[0] if len(duration_days) > 0 else 1

    # 3. Calculate offset to the nearest bolus (within a 2-hour window)
    median_offset_mins = np.nan
    if not df_bolus.empty:
        # merge_asof matches each meal with the closest bolus in time
        merged = pd.merge_asof(
            df_meals,
            df_bolus,
            left_on="meal_ts",
            right_on="bolus_ts",
            direction="nearest",
        )

        # Calculate time difference in minutes (Bolus Time - Meal Time)
        # Negative means bolus was BEFORE meal. Positive means bolus was AFTER meal.
        merged["offset_mins"] = (
            merged["bolus_ts"] - merged["meal_ts"]
        ).dt.total_seconds() / 60.0

        # Consider only boluses within +/- 120 mins to filter out unrelated correction boluses
        valid_offsets = merged.loc[merged["offset_mins"].abs() <= 120, "offset_mins"]
        if not valid_offsets.empty:
            median_offset_mins = valid_offsets.median()

    # Compile statistics
    total_meals = len(df_meals)
    meals_with_carbs = df_meals["carbs"].notna().sum()
    mean_carbs = df_meals["carbs"].mean()
    max_carbs = df_meals["carbs"].max()
    meals_per_day = total_meals / duration_days

    meal_stats_data.append(
        {
            "Release": row["Release"],
            "Patient_ID": patient_id,
            "Split": split_type,
            "Meals_Per_Day": round(meals_per_day, 1),
            "Total_Meals": total_meals,
            "Meals_With_Carbs": meals_with_carbs,
            "Mean_Carbs_(g)": round(mean_carbs, 1),
            "Max_Carbs_(g)": round(max_carbs, 1),
            "Median_Bolus_Offset_(mins)": round(median_offset_mins, 1)
            if not pd.isna(median_offset_mins)
            else None,
        }
    )

df_meals_stats = pd.DataFrame(meal_stats_data)

print("\n--- Meals and Carbohydrates Statistics ---")
display(df_meals_stats.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
context_stats_data = []

In [ ]:
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    # Define the contextual event tags we are looking for
    event_tags = ["exercise", "sleep", "work", "stress", "illness", "hypo_event"]

    patient_counts = {
        "Release": row["Release"],
        "Patient_ID": patient_id,
        "Split": split_type,
    }

    # Count the number of events for each tag
    for tag in event_tags:
        node = root.find(tag)
        count = 0
        if node is not None:
            count = len(node.findall("event"))
        patient_counts[tag.capitalize()] = count

    context_stats_data.append(patient_counts)

df_context = pd.DataFrame(context_stats_data)

print("\n--- Contextual Events (Total Counts per Split) ---")
display(df_context.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
wearable_stats_data = []

In [ ]:
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    # Common tags for 2018 Basis Band and generic acceleration
    wearable_tags = [
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_steps",
        "acceleration",
    ]

    patient_counts = {
        "Release": row["Release"],
        "Patient_ID": patient_id,
        "Split": split_type,
    }

    for tag in wearable_tags:
        node = root.find(tag)
        count = 0
        if node is not None:
            # Just counting the events to see data density
            count = len(node.findall("event"))
        patient_counts[tag] = count

    wearable_stats_data.append(patient_counts)

df_wearables = pd.DataFrame(wearable_stats_data)

print("\n--- Wearable Data Records Count ---")
display(df_wearables.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
missingness_data = []

In [ ]:
for idx, row in df_coverage.iterrows():
    patient_id = row["Patient_ID"]
    split = row["Split"]
    release = row["Release"]
    duration_days = row["Duration_Days"]

    # Calculate expected records (1 every 5 minutes = 12 per hour = 288 per day)
    expected_records = duration_days * 24 * (60 / 5)

    # Get actual CGM records from Step 3
    cgm_match = df_cgm_stats[
        (df_cgm_stats["Patient_ID"] == patient_id) & (df_cgm_stats["Split"] == split)
    ]
    actual_cgm = cgm_match["Total_Records"].values[0] if not cgm_match.empty else 0

    # Get actual Heart Rate records from Step 8 (as proxy for wearables)
    wearable_match = df_wearables[
        (df_wearables["Patient_ID"] == patient_id) & (df_wearables["Split"] == split)
    ]
    actual_hr = (
        wearable_match["basis_heart_rate"].values[0] if not wearable_match.empty else 0
    )

    # Calculate Missingness %
    # (If actual is higher than expected due to sensor overlap/duplicates, floor it to 0% missing)
    missing_cgm_pct = max(0, ((expected_records - actual_cgm) / expected_records) * 100)
    missing_hr_pct = max(0, ((expected_records - actual_hr) / expected_records) * 100)

    missingness_data.append(
        {
            "Release": release,
            "Patient_ID": patient_id,
            "Split": split,
            "Expected_Records": int(expected_records),
            "Missing_CGM_%": round(missing_cgm_pct, 1),
            "Missing_HR_%": round(missing_hr_pct, 1),
        }
    )

df_missingness = pd.DataFrame(missingness_data)

print("\n--- Missing Data Percentage (CGM vs Wearables) ---")
display(df_missingness.sort_values(by=["Release", "Patient_ID", "Split"]))

In [ ]:
hourly_data = []

In [ ]:
for idx, row in df_inventory.iterrows():
    patient_id = row["Patient_File"].split("-")[0]
    split_type = "Train" if "train" in row["Folder"].lower() else "Test"
    release = row["Release"]

    tree = ET.parse(row["Path"])
    root = tree.getroot()

    glucose_node = root.find("glucose_level")

    if glucose_node is not None:
        for event in glucose_node.findall("event"):
            ts = event.get("ts")
            val = event.get("value")
            if ts and val:
                g_val = float(val)
                # Filter out physiologically impossible values to avoid skewing the mean
                if 39 < g_val < 401:
                    dt = pd.to_datetime(ts, dayfirst=True)
                    hourly_data.append(
                        {
                            "Patient_ID": patient_id,
                            "Split": split_type,
                            "Hour": dt.hour,
                            "Glucose": g_val,
                        }
                    )

df_hourly_raw = pd.DataFrame(hourly_data)

# Aggregate by Patient, Split, and Hour to get the mean glucose
df_circadian = (
    df_hourly_raw.groupby(["Patient_ID", "Split", "Hour"])["Glucose"]
    .mean()
    .reset_index()
)

# Create a pivot table: Hours (0-23) as columns, Patients as rows
circadian_pivot = df_circadian.pivot(
    index=["Patient_ID", "Split"], columns="Hour", values="Glucose"
).round(1)

print("\n--- Mean Circadian Glucose Profile (mg/dL) by Hour of Day ---")
display(circadian_pivot)

In [ ]:
print("Consolidating EDA results into a master summary...")

# Merge DataFrames on Patient_ID and Split
df_master = df_coverage[
    ["Release", "Patient_ID", "Split", "Duration_Days", "Gap_Percentage_%"]
].copy()

# Add Clinical Metrics
df_master = df_master.merge(
    df_cgm_stats[["Patient_ID", "Split", "TIR_70_180_%", "TAR_>180_%", "TBR_<70_%"]],
    on=["Patient_ID", "Split"],
    how="left",
)

# Add Missingness Data
df_master = df_master.merge(
    df_missingness[["Patient_ID", "Split", "Missing_CGM_%", "Missing_HR_%"]],
    on=["Patient_ID", "Split"],
    how="left",
)

# Save to CSV in your Drive data folder
output_path = "/content/drive/MyDrive/GlycemIA/EDA/data/resumen_pacientes_ohio.csv"
df_master.to_csv(output_path, index=False)

print(f"✅ Master summary saved successfully at: {output_path}")
display(df_master.head(10))

# EDA Synthesis - OhioT1DM (Day 3)

## 1. Dataset Status
* **Duration and Gaps:** Most hover around 40 days in Train and 10 days in Test. Patients 559 and 591 (2018) exceed 10% sensor gaps and will require aggressive imputation.
* **Available Signals (Asymmetry):** Strict division between 2018 (Basis Band) and 2020 (Empatica). Wearable signals cannot be crossed over.
* **Missingness:** Patient 552 (Test) has a 40% loss in CGM data. This is the highest risk case for the pipeline.

## 2. Feature Decisions (Week 2)
* **Contextual Events:** `Stress` and `Illness` are discarded due to lack of density. `Sleep` is retained as a global feature. `Work` and `Exercise` will only be useful if training personalized models per patient.
* **Insulin:** `Basal` and `Temp_Basal` must be merged into a single continuous curve.
* **Temporal Derivatives:** Mandatory to create autoregressive features (e.g., moving average of the last 2 hours) to combat the *concept drift* observed in patients like 540.

## 3. Reusability of Previous EDA (Shanghai / D1NAMO - Day 2)
* The heart rate feature extraction architecture (D1NAMO) can be adapted to the 2018 Ohio cohort.
* The physiological glucose cleaning rules (<39 and >400 mg/dL are noise) validated in Shanghai apply identically here.